# 02 — Baselines & Ablation Study

Addresses reviewer points **#1 (comparative baselines)** and **#2 (ablation)**.

Every system below is evaluated on the **same five folds** of the Gold Standard
and writes out-of-fold predictions to `oof/oof_<name>.jsonl`. Notebook 03 turns
those files into the results table, the significance tests, and the error
breakdown.

### Two protocol changes from the original pipeline

**No test-fold model selection.** The original loop passed the test fold as
`eval_dataset` with `load_best_model_at_end=True` and early stopping, so the
reported epoch was chosen using the very data it was scored on. Here every
configuration trains for a fixed epoch budget with no evaluation during
training. Numbers will shift slightly against the submitted version; that shift
is the leakage being removed.

**Identical budget across configurations.** Because all variants train for the
same number of epochs, an ablation delta reflects the missing component rather
than a different amount of training.

In [ ]:
# --- identical bootstrap cell -----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers datasets seqeval pytorch-crf safetensors
!cp /content/drive/MyDrive/crypto-ner/code/*.py /content/

import sys; sys.path.append('/content')
from config import *
bootstrap()

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ROOT = /content/drive/MyDrive/crypto-ner
device: Tesla T4

  Inputs:
    [ok     ] gold_standard_bio_labeled.csv         0.9 MB  nb 01, 02, 03
    [ok     ] silver_standard_bio.csv              17.5 MB  nb 01
    [ok     ] silver_standard_dataset.csv          14.3 MB  nb 01
    [ok     ] lkb_cleaned.json                      0.2 MB  nb 01, 02, 03
    [ok     ] lkb_case_sensitive.json               0.0 MB  nb 01, 02, 03
    [ok     ] dataset_mentah_twitter.csv           16.4 MB  nb 01, funnel only

  01 → 02
    [ok     ] silver_standard_bio_naive.csv      naive silver labels, required by the -HD ablation

  01 → 03
    [ok     ] gold_strata.csv                    per-sentence stratum, used in the error breakdown

  02 → 03
    [ok     ] oof_proposed.jsonl                 out-of-fold predictions, proposed system
    [ok     ] oof_naive_silver

True

In [ ]:
import os, json
import numpy as np
import pandas as pd
import torch

import utils
from utils import (load_bio_csv, load_lkb, get_folds, run_experiment,
                   train_base_model, set_seed, get_device)

# silver_standard_bio_naive.csv comes from notebook 01 section D
assert os.path.exists(P_SILVER_BIO_NAIVE), (
    'Run notebook 01 first — it generates the naive silver set this notebook needs.')

print('device:', get_device())

device: cuda


In [ ]:
gold_ids, gold_sents, gold_labels = load_bio_csv(P_GOLD_BIO)
splits, fold_of = get_folds(len(gold_sents))

print(f'Gold sentences : {len(gold_sents)}')
for f, (tr, te) in enumerate(splits):
    print(f'  fold {f+1}: train={len(tr)}  test={len(te)}')

lkb_unamb = load_lkb(P_LKB_UNAMB)
lkb_amb   = load_lkb(P_LKB_AMB)

Gold sentences : 1461
  fold 1: train=1168  test=293
  fold 2: train=1169  test=292
  fold 3: train=1169  test=292
  fold 4: train=1169  test=292
  fold 5: train=1169  test=292


In [ ]:
# --- tier fix v2: bangun ulang dari lkb_tier_removal.json -------------------
import json
with open(P_TIER_REMOVAL, encoding='utf-8') as f:
    removal = set(json.load(f))
lkb_unamb_fixed = lkb_unamb - removal
assert 'sol' not in lkb_unamb_fixed and 'near protocol' in lkb_unamb_fixed
print(f'Unambiguous {len(lkb_unamb)} -> {len(lkb_unamb_fixed)}  ({len(removal)} dibuang)')

Unambiguous 1672 -> 1550  (179 dibuang)


## A. Base model for the −HD ablation

The proposed system's base checkpoint was pre-trained on the **denoised** silver
set. To isolate the denoising contribution we need a twin trained on the
**naive** silver set from notebook 01, with everything else held constant.

This is the single most expensive cell in the whole revision. Run it once.

In [ ]:
if not os.path.exists(os.path.join(BASE_NAIVE, 'pytorch_model.bin')):
    train_base_model(
        silver_bio_path=P_SILVER_BIO_NAIVE,
        encoder=ENCODER_TWEET,
        output_dir=BASE_NAIVE,
        use_crf=True, lr=LR_BASE, batch_size=BS_BASE, epochs=EPOCHS_BASE,
    )
else:
    print('Naive-silver base model already exists, skipping.')

Naive-silver base model already exists, skipping.


### Optional: re-train the denoised base under the same protocol OLD

The published base checkpoint used early stopping on a silver dev split. That is
**not** leakage — the dev slice came from silver, never gold — so it can be kept
as-is. Re-train only if you want the two base models to share an identical
epoch budget, which makes the −HD comparison cleaner. Recommended if compute
allows.

In [ ]:
RETRAIN_DENOISED_BASE = True   # set True for a perfectly matched -HD comparison

if RETRAIN_DENOISED_BASE:
    BASE_DENOISED = f'{CKPT}/indobertweet-crf-base-denoised-matched'   # overrides config
    if not os.path.exists(os.path.join(BASE_DENOISED, 'pytorch_model.bin')):
        train_base_model(
            silver_bio_path=P_SILVER_BIO,
            encoder=ENCODER_TWEET,
            output_dir=BASE_DENOISED,
            use_crf=True, lr=LR_BASE, batch_size=BS_BASE, epochs=EPOCHS_BASE,
        )
print('denoised base ->', BASE_DENOISED)

Silver sentences: 25726


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
100,6.607823
200,0.850495
300,0.624280
400,0.545762
500,0.407434
600,0.397860
700,0.348221
800,0.230014
900,0.207852
1000,0.191805


Base model saved -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
denoised base -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched


### A3a

In [ ]:
# --- pemeriksaan sebelum membakar 35 menit GPU -----------------------------
from utils import extract_entities

assert os.path.exists(P_SILVER_BIO_V2), 'silver v2 tidak ada — jalankan A2 di nb 01'
_, s2, l2 = load_bio_csv(P_SILVER_BIO_V2)
n2 = sum(len(extract_entities(l)) for l in l2)
print(f'silver v2 : {len(s2)} kalimat, {n2} entitas')
assert (len(s2), n2) == (25726, 22892), 'file v2 bukan hasil A2 — periksa ulang'

assert os.path.exists(os.path.join(BASE_NAIVE, 'pytorch_model.bin')), 'BASE_NAIVE hilang'
print('BASE_NAIVE ada — tidak akan dilatih ulang, ablation -HD tetap punya kontrolnya')

old_ckpt = f'{CKPT}/indobertweet-crf-base-denoised-matched'
print('checkpoint v1 masih utuh :', os.path.exists(os.path.join(old_ckpt, 'pytorch_model.bin')))
print('target v2                :', BASE_DENOISED_V2)
assert not os.path.exists(os.path.join(BASE_DENOISED_V2, 'pytorch_model.bin')), \
    'base v2 sudah terlatih — lewati A3'

silver v2 : 25726 kalimat, 22892 entitas
BASE_NAIVE ada — tidak akan dilatih ulang, ablation -HD tetap punya kontrolnya
checkpoint v1 masih utuh : True
target v2                : /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2


### RETRAIN DENOISED BASE (V2)

In [ ]:
RETRAIN_DENOISED_BASE = True

if RETRAIN_DENOISED_BASE:
    BASE_DENOISED = BASE_DENOISED_V2                      # tier eksklusif, silver v2
    if not os.path.exists(os.path.join(BASE_DENOISED, 'pytorch_model.bin')):
        train_base_model(
            silver_bio_path=P_SILVER_BIO_V2,              # <- satu-satunya perubahan nyata
            encoder=ENCODER_TWEET,
            output_dir=BASE_DENOISED,
            use_crf=True, lr=LR_BASE, batch_size=BS_BASE, epochs=EPOCHS_BASE,
        )
    else:
        print('base v2 sudah ada, pelatihan dilewati')

print('denoised base ->', BASE_DENOISED)

Silver sentences: 25726


config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/235k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Step,Training Loss
100,5.819244
200,0.791066
300,0.583513
400,0.560440
500,0.420943
600,0.418188
700,0.346100
800,0.275088
900,0.236427
1000,0.188776


Base model saved -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
denoised base -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2


### BASE v1

In [ ]:
# --- kembalikan base ke v1 dan pastikan patch seed aktif -------------------
import inspect
from utils import _training_args, run_experiment

BASE_DENOISED = f'{CKPT}/indobertweet-crf-base-denoised-matched'
assert os.path.exists(os.path.join(BASE_DENOISED, 'pytorch_model.bin'))
print('BASE_DENOISED ->', BASE_DENOISED)

# patch seed harus ada di utils.py yang benar-benar termuat
assert 'seed' in inspect.signature(_training_args).parameters, \
    'utils.py di Drive masih versi lama — salin ulang lalu restart runtime'
src = inspect.getsource(run_experiment)
assert 'config.get("seed", 42) + f' in src, 'patch set_seed belum aktif'
print('patch seed aktif')

# CONFIGS harus dibangun ulang dengan base v1 dan LKB lama
for c in CONFIGS:
    if c.get('init_from'):
        print(f"  {c['name']:20s} <- {c['init_from']}")
gz = next(c for c in CONFIGS if c['name'] == 'gazetteer')
print('  gazetteer unamb size:', len(gz['lkb_unambiguous']))

BASE_DENOISED -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
patch seed aktif
  base_only            <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
  naive_silver_ft      <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-naive
  proposed             <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
  gazetteer unamb size: 1672


## B. Configurations

Several rows do double duty: `indobertweet_crf_gold` is simultaneously a
transformer baseline **and** the −DS ablation, and `base_only` is both the
Table I base model **and** the −FT ablation. That overlap is what lets the paper
answer reviewer #1 and #2 with a single table instead of two.

| # | Config | Role |
|---|---|---|
| 1 | `gazetteer` | Baseline — dictionary matching only, no learning |
| 2 | `bilstm_crf` | Baseline — non-transformer neural |
| 3 | `mbert_crf_gold` | Baseline — multilingual transformer |
| 4 | `indobert_crf_gold` | Baseline — isolates domain-adaptive pretraining |
| 5 | `indobertweet_crf_gold` | Baseline + **ablation −DS** |
| 6 | `base_only` | Table I + **ablation −FT** |
| 7 | `naive_silver_ft` | **Ablation −HD** |
| 8 | `softmax_ft` | **Ablation −CRF** |
| 9 | `proposed` | Full system |

In [ ]:
# --- arsipkan OOF yang bergantung pada tier lama ---------------------------
import shutil
ARCHIVE = f'{OOF}/v1_tier'
os.makedirs(ARCHIVE, exist_ok=True)

for name in ['gazetteer', 'base_only', 'proposed']:
    src = f'{OOF}/oof_{name}.jsonl'
    dst = f'{ARCHIVE}/oof_{name}.jsonl'
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f'arsip  {name:12s} -> {dst}')
    else:
        print(f'sudah tidak ada: {name}')

print('\nOOF tersisa:')
for f in sorted(os.listdir(OOF)):
    if f.startswith('oof_'):
        print('  ', f)

arsip  gazetteer    -> /content/drive/MyDrive/crypto-ner/oof/v1_tier/oof_gazetteer.jsonl
arsip  base_only    -> /content/drive/MyDrive/crypto-ner/oof/v1_tier/oof_base_only.jsonl
arsip  proposed     -> /content/drive/MyDrive/crypto-ner/oof/v1_tier/oof_proposed.jsonl

OOF tersisa:
   oof_base_only_OLD.jsonl
   oof_bilstm_crf.jsonl
   oof_indobert_crf_gold.jsonl
   oof_indobertweet_crf_gold.jsonl
   oof_mbert_crf_gold.jsonl
   oof_naive_silver_ft.jsonl
   oof_proposed_published.jsonl
   oof_softmax_ft.jsonl


In [ ]:
CONFIGS = [
    # tier aditif (v1). Varian tier eksklusif: lihat lkb_tier_removal.json,
    # hasilnya dilaporkan di limitations, bukan di tabel utama.
    dict(name='gazetteer', type='gazetteer',
         lkb_unambiguous=lkb_unamb, lkb_ambiguous=lkb_amb, denoise=True),

    dict(name='bilstm_crf', type='bilstm',
         use_crf=True, epochs=20, lr=1e-3),

    dict(name='mbert_crf_gold', type='transformer', encoder=ENCODER_MBERT,
         use_crf=True, init_from=None, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),

    dict(name='indobert_crf_gold', type='transformer', encoder=ENCODER_INDOB,
         use_crf=True, init_from=None, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),

    # -DS : no distant supervision, gold only
    dict(name='indobertweet_crf_gold', type='transformer', encoder=ENCODER_TWEET,
         use_crf=True, init_from=None, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),

    # -FT : silver-trained base model, never fine-tuned on gold
    dict(name='base_only', type='pretrained_only', encoder=ENCODER_TWEET,
         use_crf=True, init_from=BASE_DENOISED),

    # -HD : base pre-trained on naive (non-denoised) silver
    dict(name='naive_silver_ft', type='transformer', encoder=ENCODER_TWEET,
         use_crf=True, init_from=BASE_NAIVE, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),

    # -CRF : softmax head instead of CRF
    dict(name='softmax_ft', type='transformer', encoder=ENCODER_TWEET,
         use_crf=False, init_from=None, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),

    # full proposed system
    dict(name='proposed', type='transformer', encoder=ENCODER_TWEET,
         use_crf=True, init_from=BASE_DENOISED, lr=LR_FT, batch_size=BS_FT, epochs=EPOCHS_FT),
]

print(f'{len(CONFIGS)} configurations queued')

9 configurations queued


#### 3 seeds

In [ ]:
SEEDS = [1337, 2024]          # seed 42 sudah ada sebagai run utama
SEED_CONFIGS = []
for s in SEEDS:
    for base_name in ['proposed', 'naive_silver_ft']:
        cfg = dict(next(c for c in CONFIGS if c['name'] == base_name))
        cfg['name'] = f'{base_name}_s{s}'
        cfg['seed'] = s
        SEED_CONFIGS.append(cfg)

for cfg in SEED_CONFIGS:
    print(f"{cfg['name']:24s} seed={cfg['seed']}  init_from={cfg.get('init_from')}")
assert len(SEED_CONFIGS) == 4

proposed_s1337           seed=1337  init_from=/content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
naive_silver_ft_s1337    seed=1337  init_from=/content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-naive
proposed_s2024           seed=2024  init_from=/content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched
naive_silver_ft_s2024    seed=2024  init_from=/content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-naive


In [ ]:
cfg = SEED_CONFIGS[0]          # proposed_s1337
run_experiment(cfg, gold_sents, gold_labels, gold_ids, splits, out_dir=OOF)
torch.cuda.empty_cache()

from utils import load_oof
a = load_oof(f'{OOF}/oof_proposed.jsonl')
b = load_oof(f'{OOF}/oof_proposed_s1337.jsonl')
d = sum(1 for x, y in zip(a, b) if x['pred'] != y['pred'])
print(f'{d}/{len(a)} kalimat berbeda prediksinya')
assert d > 0, 'SEED TIDAK BERPENGARUH — patch belum aktif, hentikan run'


=== proposed_s1337 (transformer) ===
  fold 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,1.641484


  fold 2/5


...

  fold 5/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,2.073251


  pooled  P=0.8416 R=0.8807 F1=0.8607
  per-fold F1 mean=0.8599 (± 0.0323)  -> /content/drive/MyDrive/crypto-ner/oof/oof_proposed_s1337.jsonl
64/1461 kalimat berbeda prediksinya


In [ ]:
for cfg in SEED_CONFIGS[1:]:
    out_path = os.path.join(OOF, f"oof_{cfg['name']}.jsonl")
    if os.path.exists(out_path):
        print(f"[skip] {cfg['name']}");  continue
    try:
        run_experiment(cfg, gold_sents, gold_labels, gold_ids, splits, out_dir=OOF)
    except Exception as e:
        print(f"[FAIL] {cfg['name']}: {type(e).__name__}: {e}")
    torch.cuda.empty_cache()


=== naive_silver_ft_s1337 (transformer) ===
  fold 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,2.580661


  fold 2/5


...

  fold 5/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,2.640966


  pooled  P=0.8218 R=0.8739 F1=0.8471
  per-fold F1 mean=0.8466 (± 0.0284)  -> /content/drive/MyDrive/crypto-ner/oof/oof_naive_silver_ft_s2024.jsonl


### A note on `softmax_ft`

The −CRF variant is initialised from the encoder rather than from
`BASE_DENOISED`, because the saved base checkpoint contains CRF transition
parameters that a softmax model has no slot for. To ablate the CRF layer *with*
distant supervision, first train a softmax base model on the denoised silver set
(`use_crf=False` in `train_base_model`) and point `init_from` at it.

Either way, state in the caption which variant you ran. Reviewers accept the
cheaper version as long as it is described accurately.

In [ ]:
print('BASE_DENOISED ->', BASE_DENOISED)
assert '-v2' in BASE_DENOISED, 'CONFIGS masih memegang checkpoint lama!'
print(sorted(os.listdir(BASE_DENOISED)))      # harus ada pytorch_model.bin

for c in CONFIGS:
    if c.get('init_from'):
        print(f"  {c['name']:24s} <- {c['init_from']}")

BASE_DENOISED -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
['pytorch_model.bin', 'tokenizer.json', 'tokenizer_config.json']
  base_only                <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
  naive_silver_ft          <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-naive
  proposed                 <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2


In [ ]:
print('BASE_DENOISED ->', BASE_DENOISED)
assert BASE_DENOISED.endswith('-v2'), 'CONFIGS masih memegang checkpoint lama!'
print(sorted(os.listdir(BASE_DENOISED)))

for c in CONFIGS:
    if c.get('init_from'):
        print(f"  {c['name']:24s} <- {c['init_from']}")

gz = next(c for c in CONFIGS if c['name'] == 'gazetteer')
assert 'sol' not in gz['lkb_unambiguous'], 'gazetteer masih memakai tier lama'
assert len(gz['lkb_unambiguous']) == len(lkb_unamb_fixed)
print('gazetteer memakai tier v2 — ok')

BASE_DENOISED -> /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
['pytorch_model.bin', 'tokenizer.json', 'tokenizer_config.json']
  base_only                <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
  naive_silver_ft          <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-naive
  proposed                 <- /content/drive/MyDrive/crypto-ner/ckpt/indobertweet-crf-base-denoised-matched-v2
gazetteer memakai tier v2 — ok


## C. Run everything

Each configuration is skipped if its OOF file already exists, so the cell is
safe to re-run after a Colab disconnect.

In [ ]:
results = []
for cfg in CONFIGS:
    out_path = os.path.join(OOF, f"oof_{cfg['name']}.jsonl")
    if os.path.exists(out_path):
        print(f"[skip] {cfg['name']} already done -> {out_path}")
        continue
    try:
        res = run_experiment(cfg, gold_sents, gold_labels, gold_ids, splits, out_dir=OOF)
        results.append(res)
    except Exception as e:
        print(f"[FAIL] {cfg['name']}: {type(e).__name__}: {e}")
    torch.cuda.empty_cache()


=== gazetteer (gazetteer) ===
  pooled  P=0.7993 R=0.7716 F1=0.7852
  per-fold F1 mean=0.7839 (± 0.0500)  -> /content/drive/MyDrive/crypto-ner/oof/oof_gazetteer.jsonl
[skip] bilstm_crf already done -> /content/drive/MyDrive/crypto-ner/oof/oof_bilstm_crf.jsonl
[skip] mbert_crf_gold already done -> /content/drive/MyDrive/crypto-ner/oof/oof_mbert_crf_gold.jsonl
[skip] indobert_crf_gold already done -> /content/drive/MyDrive/crypto-ner/oof/oof_indobert_crf_gold.jsonl
[skip] indobertweet_crf_gold already done -> /content/drive/MyDrive/crypto-ner/oof/oof_indobertweet_crf_gold.jsonl

=== base_only (pretrained_only) ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  pooled  P=0.8059 R=0.7657 F1=0.7852
  per-fold F1 mean=0.7843 (± 0.0432)  -> /content/drive/MyDrive/crypto-ner/oof/oof_base_only.jsonl
[skip] naive_silver_ft already done -> /content/drive/MyDrive/crypto-ner/oof/oof_naive_silver_ft.jsonl
[skip] softmax_ft already done -> /content/drive/MyDrive/crypto-ner/oof/oof_softmax_ft.jsonl

=== proposed (transformer) ===
  fold 1/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

  fold 5/5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Step,Training Loss
100,1.159605


  pooled  P=0.8261 R=0.8638 F1=0.8445
  per-fold F1 mean=0.8440 (± 0.0318)  -> /content/drive/MyDrive/crypto-ner/oof/oof_proposed.jsonl


In [ ]:
from utils import load_oof, entity_f1

for name in ['gazetteer', 'base_only', 'proposed']:
    new = load_oof(f'{OOF}/oof_{name}.jsonl')
    old = load_oof(f'{OOF}/v1_tier/oof_{name}.jsonl')
    d = sum(1 for a, b in zip(new, old) if a['pred'] != b['pred'])
    m_new = entity_f1([r['true'] for r in new], [r['pred'] for r in new])
    m_old = entity_f1([r['true'] for r in old], [r['pred'] for r in old])
    print(f'{name:12s} F1 {m_old["f1"]:.4f} -> {m_new["f1"]:.4f}   '
          f'P {m_old["precision"]:.4f} -> {m_new["precision"]:.4f}   '
          f'R {m_old["recall"]:.4f} -> {m_new["recall"]:.4f}   '
          f'({d} kalimat berbeda)')

# kontrol: -HD tidak boleh ikut bergerak
nai = load_oof(f'{OOF}/oof_naive_silver_ft.jsonl')
print(f'\nnaive_silver_ft F1 {entity_f1([r["true"] for r in nai], [r["pred"] for r in nai])["f1"]:.4f}'
      f'   (harus tetap 0.8416)')

gazetteer    F1 0.5922 -> 0.7852   P 0.4691 -> 0.7993   R 0.8029 -> 0.7716   (605 kalimat berbeda)
base_only    F1 0.5905 -> 0.7852   P 0.4672 -> 0.8059   R 0.8020 -> 0.7657   (627 kalimat berbeda)
proposed     F1 0.8541 -> 0.8445   P 0.8328 -> 0.8261   R 0.8765 -> 0.8638   (164 kalimat berbeda)

naive_silver_ft F1 0.8416   (harus tetap 0.8416)


In [ ]:
from utils import load_oof, entity_f1, paired_bootstrap, approximate_randomization

def P(path):
    r = load_oof(path)
    return [x['true'] for x in r], [x['pred'] for x in r]

y,  p_v2  = P(f'{OOF}/oof_proposed.jsonl')
_,  p_v1  = P(f'{OOF}/v1_tier/oof_proposed.jsonl')
_,  p_nai = P(f'{OOF}/oof_naive_silver_ft.jsonl')

for lbl, a, b in [('proposed v2 vs proposed v1', p_v2, p_v1),
                  ('proposed v2 vs -HD',         p_v2, p_nai),
                  ('proposed v1 vs -HD',         p_v1, p_nai)]:
    d  = entity_f1(y, a)['f1'] - entity_f1(y, b)['f1']
    pb = paired_bootstrap(y, a, b, n_resamples=100000)
    ar = approximate_randomization(y, a, b, n_trials=100000)
    print(f'{lbl:26s} dF1 {d:+.4f}   bootstrap {pb}   AR {ar}')

proposed v2 vs proposed v1 dF1 -0.0096   bootstrap {'delta_f1': 0.009581204991145564, 'p_value': 0.16327, 'ci_low': -0.003930553016302163, 'ci_high': 0.023123340301188323}   AR {'p_value': 0.16453835461645383}
proposed v2 vs -HD         dF1 +0.0029   bootstrap {'delta_f1': -0.002870627159052286, 'p_value': 0.66534, 'ci_low': -0.015870226917113876, 'ci_high': 0.010096579237451725}   AR {'p_value': 0.6679033209667903}
proposed v1 vs -HD         dF1 +0.0125   bootstrap {'delta_f1': -0.01245183215019785, 'p_value': 0.02576, 'ci_low': -0.02340894722640802, 'ci_high': -0.00157625525854436}   AR {'p_value': 0.024329756702432977}


In [ ]:
import shutil, os
V2DIR = f'{OOF}/v2_tier'; os.makedirs(V2DIR, exist_ok=True)
for name in ['gazetteer', 'base_only', 'proposed']:
    shutil.move(f'{OOF}/oof_{name}.jsonl', f'{V2DIR}/oof_{name}.jsonl')
    shutil.move(f'{OOF}/v1_tier/oof_{name}.jsonl', f'{OOF}/oof_{name}.jsonl')
    print(f'{name}: v2 -> v2_tier, v1 dipulihkan')
os.rmdir(f'{OOF}/v1_tier')

from utils import load_oof, entity_f1
for name in ['gazetteer', 'base_only', 'proposed', 'naive_silver_ft']:
    r = load_oof(f'{OOF}/oof_{name}.jsonl')
    print(f'{name:18s} {entity_f1([x["true"] for x in r], [x["pred"] for x in r])["f1"]:.4f}')

gazetteer: v2 -> v2_tier, v1 dipulihkan
base_only: v2 -> v2_tier, v1 dipulihkan
proposed: v2 -> v2_tier, v1 dipulihkan
gazetteer          0.5922
base_only          0.5905
proposed           0.8541
naive_silver_ft    0.8416


In [ ]:
# Quick summary across whatever finished. Notebook 03 produces the paper table.
from utils import load_oof, entity_f1

rows = []
for cfg in CONFIGS:
    p = os.path.join(OOF, f"oof_{cfg['name']}.jsonl")
    if not os.path.exists(p):
        continue
    recs = load_oof(p)
    y_true = [r['true'] for r in recs]
    y_pred = [r['pred'] for r in recs]
    folds  = np.array([r['fold'] for r in recs])
    per_fold = [entity_f1([y_true[i] for i in np.where(folds == f)[0]],
                          [y_pred[i] for i in np.where(folds == f)[0]])['f1']
                for f in sorted(set(folds))]
    pooled = entity_f1(y_true, y_pred)
    rows.append({'System': cfg['name'],
                 'P (pooled)': round(pooled['precision'], 4),
                 'R (pooled)': round(pooled['recall'], 4),
                 'F1 (pooled)': round(pooled['f1'], 4),
                 'F1 mean±std (folds)': f"{np.mean(per_fold):.4f} ± {np.std(per_fold):.4f}"})

df_sum = pd.DataFrame(rows).sort_values('F1 (pooled)', ascending=False)
df_sum.to_csv(f'{OOF}/summary_quick.csv', index=False)
display(df_sum)

,System,P (pooled),R (pooled),F1 (pooled),F1 mean±std (folds)
3,indobert_crf_gold,0.8561,0.8553,0.8557,0.8554 ± 0.0106
8,proposed,0.8328,0.8765,0.8541,0.8533 ± 0.0311
6,naive_silver_ft,0.8191,0.8655,0.8416,0.8412 ± 0.0359
2,mbert_crf_gold,0.8227,0.8325,0.8276,0.8272 ± 0.0269
7,softmax_ft,0.8209,0.7910,0.8057,0.8050 ± 0.0199
4,indobertweet_crf_gold,0.8454,0.7631,0.8021,0.8010 ± 0.0267
1,bilstm_crf,0.7887,0.6252,0.6975,0.6980 ± 0.0323
0,gazetteer,0.4691,0.8029,0.5922,0.5923 ± 0.0461
5,base_only,0.4672,0.8020,0.5905,0.5908 ± 0.0405


### If you have to cut scope

Compute budget is finite. Drop in this order:

1. `bilstm_crf` — three transformer baselines already satisfy "several NER models"
2. `mbert_crf_gold`

Never drop `naive_silver_ft`: it is the only direct evidence for the paper's
headline contribution. `indobert_crf_gold` is the second-most important row,
since without it the choice of IndoBERTweet over IndoBERT is unjustified.